In [1]:

from GenZ import prefill_moddeling, decode_moddeling, chunked_moddeling
from GenZ.system import System
from copy import deepcopy

In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
## Library and inputs
import plotly.express as px
import pandas as pd
from plotly.subplots import make_subplots
import plotly.graph_objects as go
import numpy as np

import plotnine as p9
from plotnine import ggplot, aes, geom_point, geom_line, labs, element_text

In [4]:
# Switch Switch Ring 8 NV - 8 Nv - 4 eth
tp, pp, network_config = (64, 4,
    {
    "topology": ["Switch", "Switch", "Ring"],
    "npus_count": [8, 8, 4],
    "Type" : ['NV', 'NV', 'eth'],
    },
    )

In [5]:
Network_latency = {"NV": 500, "Infi": 10000, "eth": 10000, "Optical": 200}
Network_bws = {"NV": 1800, "Infi": 256, "eth": 256, "Optical": 900}

In [8]:
model, total_batch_size, input_tokens, output_tokens = ('Hypothetical/SuperLLM-5T-Dense', 32, 32000, 1000)

platform_config = System(flops=9000, off_chip_mem_size=256*1024, offchip_mem_bw=13500,
interchip_link_bw=900, interchip_link_latency=0.25, external_mem_bw=1e-6)
icn_type = network_config['Type']
Latency = []
BW = []
for icn in icn_type:
    Latency.append(Network_latency[icn])
    BW.append(Network_bws[icn])
network_config.update({"latency": Latency})
network_config.update({"bandwidth": BW})
print(f"System: Model: {model}, Batch: {total_batch_size}, TP/PP:{tp},{pp}")
# Prefill
prefill_outputs = prefill_moddeling(model = model, batch_size = total_batch_size*pp,
                        input_tokens = input_tokens,
                        system_name = deepcopy(platform_config),
                        bits='int8', model_offload= False,
                        collective_strategy='ASTRA-SIM', network_config=network_config,
                        parallelism_hierarchy = f"TP{{{tp}}}_PP{{{pp}}}",
                        tensor_parallel = tp, pipeline_parallel = pp, debug=False)

System: Model: Hypothetical/SuperLLM-5T-Dense, Batch: 32, TP/PP:64,4


ModuleNotFoundError: No module named 'chakra'